# Eliminador de fondo negro
Ejecuta la celda siguiente, ingresa la ruta del archivo de imagen (JPG, PNG, etc.) y el script generará una copia en formato PNG con el fondo negro vuelto transparente. Puedes ajustar la tolerancia para abarcar tonos cercanos al negro.


In [ ]:
from pathlib import Path
from typing import Optional, Union

try:
    from PIL import Image
except ImportError as exc:  # pragma: no cover
    raise SystemExit("Necesitas instalar Pillow antes de continuar. Ejecuta `pip install pillow`.") from exc

try:
    import tkinter as tk
    from tkinter import filedialog
except Exception:  # pragma: no cover
    tk = None
    filedialog = None

FILE_DIALOG_ENABLED = filedialog is not None


def remove_black_background(image_path: Union[str, Path], tolerance: int = 25) -> Path:
    '''Convierte en transparente cualquier píxel cuyo color esté dentro del umbral de negro.'''
    resolved_path = Path(image_path).expanduser().resolve()
    if not resolved_path.exists():
        raise FileNotFoundError(f"No se encontró la imagen en {resolved_path}")

    with Image.open(resolved_path) as original_image:
        image = original_image.convert("RGBA")

    pixels = image.getdata()
    new_pixels = []
    clamped_threshold = max(0, min(tolerance, 255))

    for red, green, blue, alpha in pixels:
        if red <= clamped_threshold and green <= clamped_threshold and blue <= clamped_threshold:
            new_pixels.append((red, green, blue, 0))
        else:
            new_pixels.append((red, green, blue, alpha))

    image.putdata(new_pixels)
    output_path = resolved_path.with_name(f"{resolved_path.stem}_sin_fondo.png")
    image.save(output_path)
    image.close()
    return output_path


def pick_image_with_dialog() -> Optional[Path]:
    if not FILE_DIALOG_ENABLED:
        return None
    try:
        root = tk.Tk()
        root.withdraw()
        root.update()
        selected = filedialog.askopenfilename(
            title="Selecciona la imagen",
            filetypes=[
                ("Archivos de imagen", "*.png;*.jpg;*.jpeg;*.bmp;*.tif;*.tiff"),
                ("Todos los archivos", "*.*"),
            ],
        )
        root.destroy()
    except Exception:
        return None
    if not selected:
        return None
    candidate = Path(selected).expanduser()
    return candidate if candidate.exists() else None


def prompt_for_image() -> Path:
    if not FILE_DIALOG_ENABLED:
        print("El selector gráfico no está disponible en este entorno. Pega la ruta manualmente.")

    while True:
        if FILE_DIALOG_ENABLED:
            typed_path = input(
                "Presiona Enter para abrir el explorador o pega la ruta manualmente: "
            ).strip()
            if not typed_path:
                selected = pick_image_with_dialog()
                if selected:
                    print(f"Usaré la imagen: {selected}")
                    return selected
                print("No seleccionaste ningún archivo. Intenta otra vez o pega la ruta manualmente.")
                continue
        else:
            typed_path = input("Ruta del archivo de imagen: ").strip()

        if not typed_path:
            print("Necesito una ruta válida. Intenta de nuevo.")
            continue

        candidate = Path(typed_path).expanduser()
        if candidate.exists():
            return candidate
        print(f"No encontré '{candidate}'. Verifica la ruta e inténtalo nuevamente.")


def prompt_for_tolerance(default: int = 25) -> int:
    while True:
        user_input = input(f"Tolerancia (0-255, Enter = {default}): ").strip()
        if not user_input:
            return default
        try:
            value = int(user_input)
        except ValueError:
            print("Introduce un número entero o deja vacío para usar el valor por defecto.")
            continue
        if 0 <= value <= 255:
            return value
        print("El valor debe estar entre 0 y 255.")


def main():
    print("=== Eliminador de fondo negro ===")
    image_path = prompt_for_image()
    tolerance = prompt_for_tolerance()
    output_path = remove_black_background(image_path, tolerance)
    print(f"Listo: guardé la imagen con transparencia en {output_path}")


main()


=== Eliminador de fondo negro ===
